# Niko AI — YOLOv8 Crop Disease Classifier

**Dataset:** New Plant Diseases Dataset (87,000+ images, 38 classes)  
**Model:** YOLOv8 Classification

### Instructions
1. Set runtime: `Runtime → Change runtime type → T4 GPU`
2. Run **Cell 0** first — it defines all shared variables
3. Run remaining cells top to bottom
4. After any kernel restart, **re-run Cell 0** before anything else


## Cell 0 — Run This First (and after every kernel restart)

Defines `IN_COLAB`, `DRIVE_PROJECT_PATH`, `DATASET_DIR`, `WEIGHTS_OUTPUT_DIR`.

In [ ]:
import sys
from pathlib import Path

# ── Detect environment ───────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Environment : {"Google Colab" if IN_COLAB else "Local Jupyter"}')

# ── Paths ────────────────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    DRIVE_PROJECT_PATH = Path('/content/drive/MyDrive/niko-ai')  # <- change if needed
    DATASET_DIR        = Path('/content/dataset')

else:
    DRIVE_PROJECT_PATH = Path(r'c:/Jayesh/projects/miko')          # <- your local project
    DATASET_DIR        = Path(r'c:/Jayesh/projects/miko/ai-model/dataset')

WEIGHTS_OUTPUT_DIR = DRIVE_PROJECT_PATH / 'ai-model' / 'weights'
WEIGHTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project     : {DRIVE_PROJECT_PATH}')
print(f'Dataset dir : {DATASET_DIR}')
print(f'Weights out : {WEIGHTS_OUTPUT_DIR}')
print()
print('Cell 0 complete. All variables are set.')

## Step 1 — Install Dependencies

> In Colab the kernel restarts after this cell. **Re-run Cell 0 immediately after restart**, then continue from Step 2.

In [ ]:
import subprocess, sys

# Check if compatible version already installed — skip if so
needs_install = True
try:
    import ultralytics
    from packaging.version import Version
    if Version(ultralytics.__version__) >= Version('8.3.0'):
        needs_install = False
        print(f'ultralytics {ultralytics.__version__} already OK. Skipping install.')
except Exception:
    pass

if needs_install:
    # ultralytics >= 8.3.0 fixes the PyTorch 2.6 UnpicklingError
    # Never reinstall torch in Colab — it would break the CUDA build
    try:
        IN_COLAB
    except NameError:
        try:
            import google.colab
            IN_COLAB = True
        except ImportError:
            IN_COLAB = False

    pkgs = ['ultralytics>=8.3.0', 'kaggle'] if IN_COLAB else \
           ['ultralytics>=8.3.0', 'kaggle', 'torch', 'torchvision']

    print(f'Installing: {pkgs} ...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install'] + pkgs + ['--quiet'],
        capture_output=True, text=True
    )
    print('STDERR:', r.stderr[-500:] if r.returncode != 0 else 'none')
    print('Install complete.' if r.returncode == 0 else 'Install FAILED.')

    if IN_COLAB:
        print()
        print('Kernel restarting now...')
        print('After restart: re-run Cell 0, then continue from Step 2.')
        import os
        os.kill(os.getpid(), 9)

import torch, ultralytics
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
elif IN_COLAB:
    print('No GPU — Runtime > Change runtime type > T4 GPU')
ultralytics.checks()

## Step 2 — Kaggle Credentials

In [ ]:
# Guard: re-define IN_COLAB and DRIVE_PROJECT_PATH if Cell 0 was not run
import sys
from pathlib import Path

try:
    IN_COLAB
    DRIVE_PROJECT_PATH
except NameError:
    print('WARNING: Cell 0 was not run. Running it now...')
    try:
        import google.colab
        IN_COLAB = True
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_PROJECT_PATH = Path('/content/drive/MyDrive/niko-ai')
        DATASET_DIR        = Path('/content/dataset')
    except ImportError:
        IN_COLAB = False
        DRIVE_PROJECT_PATH = Path(r'c:/Jayesh/projects/miko')
        DATASET_DIR        = Path(r'c:/Jayesh/projects/miko/ai-model/dataset')
    WEIGHTS_OUTPUT_DIR = DRIVE_PROJECT_PATH / 'ai-model' / 'weights'
    WEIGHTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import os, json, shutil

local_kaggle = Path.home() / '.kaggle' / 'kaggle.json'

if local_kaggle.exists():
    print(f'Kaggle credentials already at {local_kaggle}')

elif IN_COLAB and (DRIVE_PROJECT_PATH / 'kaggle.json').exists():
    local_kaggle.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(DRIVE_PROJECT_PATH / 'kaggle.json'), str(local_kaggle))
    local_kaggle.chmod(0o600)
    print(f'Credentials loaded from Drive.')

else:
    KAGGLE_USERNAME = ''  # <- fill in your Kaggle username
    KAGGLE_KEY      = ''  # <- fill in your Kaggle API key
    # Get from: https://www.kaggle.com/settings -> API -> Create New Token

    if KAGGLE_USERNAME and KAGGLE_KEY:
        local_kaggle.parent.mkdir(parents=True, exist_ok=True)
        with open(local_kaggle, 'w') as f:
            json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
        local_kaggle.chmod(0o600)
        print('Credentials saved.')
    else:
        print('ACTION NEEDED: Fill in KAGGLE_USERNAME and KAGGLE_KEY above.')
        print('Get your key at: https://www.kaggle.com/settings -> API -> Create New Token')

## Step 3 — Download Dataset

In [ ]:
try: DATASET_DIR
except NameError: exec(open('cell_globals') if False else '') or exec("""
from pathlib import Path
try:
    import google.colab; IN_COLAB=True; DATASET_DIR=Path('/content/dataset')
except: IN_COLAB=False; DATASET_DIR=Path('c:/Jayesh/projects/miko/ai-model/dataset')
""")

import os
already = DATASET_DIR.exists() and any(True for _ in DATASET_DIR.rglob('train'))
if already:
    print(f'Dataset already at {DATASET_DIR} — skipping.')
else:
    print('Downloading (~3 GB, takes a few minutes)...')
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    ret = os.system(
        f'kaggle datasets download -d vipoooool/new-plant-diseases-dataset '
        f'-p "{DATASET_DIR}" --unzip'
    )
    print('Done.' if ret == 0 else 'FAILED — check Kaggle credentials in Step 2.')

## Step 4 — Explore Dataset

In [ ]:
import pandas as pd
from pathlib import Path

try: DATASET_DIR
except NameError: DATASET_DIR = Path('/content/dataset')

def find_split(base, split):
    hits = [d for d in base.rglob(split)
            if d.is_dir() and any(x.is_dir() for x in d.iterdir())]
    return sorted(hits)[0] if hits else None

TRAIN_DIR = find_split(DATASET_DIR, 'train')
VALID_DIR  = find_split(DATASET_DIR, 'valid')

if TRAIN_DIR is None:
    raise FileNotFoundError(
        f'No train/ directory found in {DATASET_DIR}.\n'
        'Check Step 3 completed successfully.'
    )

DATA_DIR = TRAIN_DIR.parent
print(f'TRAIN : {TRAIN_DIR}')
print(f'VALID : {VALID_DIR}')
print(f'DATA  : {DATA_DIR}')

classes = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
rows = [{'class': c,
         'images': len(list((TRAIN_DIR/c).glob('*.jpg'))) +
                   len(list((TRAIN_DIR/c).glob('*.JPG')))}
        for c in classes]
df = pd.DataFrame(rows)
print(f'\nClasses: {len(classes)} | Total: {df.images.sum():,} images')
print(df.to_string(index=False))

## Step 5 — Sample Image Grid

In [ ]:
import random, matplotlib.pyplot as plt, matplotlib.image as mpimg

sample = random.sample(classes, min(12, len(classes)))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, cls in zip(axes.flatten(), sample):
    imgs = list((TRAIN_DIR/cls).glob('*.jpg')) + list((TRAIN_DIR/cls).glob('*.JPG'))
    if imgs:
        ax.imshow(mpimg.imread(str(random.choice(imgs))))
        ax.set_title(cls.replace('___', '\n'), fontsize=7)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

## Step 6 — Configure Training

In [ ]:
import torch
from ultralytics import YOLO

EPOCHS     = 50   # @param {type:'integer'}
BATCH_SIZE = 32   # @param {type:'integer'}
IMAGE_SIZE = 224  # @param {type:'integer'}
MODEL_SIZE = 'n'  # @param ['n','s','m','l','x'] {type:'string'}
PROJECT    = 'niko_ai_training'
RUN        = 'run1'

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
gpu_info = torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'

print(f'Model      : YOLOv8{MODEL_SIZE}-cls')
print(f'Epochs     : {EPOCHS}')
print(f'Batch      : {BATCH_SIZE}')
print(f'Image size : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Device     : {gpu_info}')
print(f'Data       : {DATA_DIR}')

## Step 7 — Train

In [ ]:
model = YOLO(f'yolov8{MODEL_SIZE}-cls.pt')

results = model.train(
    data          = str(DATA_DIR),
    epochs        = EPOCHS,
    batch         = BATCH_SIZE,
    imgsz         = IMAGE_SIZE,
    project       = PROJECT,
    name          = RUN,
    exist_ok      = True,
    save          = True,
    save_period   = 10,
    patience      = 15,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,
    augment       = True,
    degrees       = 10,
    flipud        = 0.3,
    fliplr        = 0.5,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    device        = DEVICE,
    verbose       = True,
    plots         = True,
)

LOCAL_WEIGHTS = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nTraining complete. Weights: {LOCAL_WEIGHTS}')

## Step 8 — Validate

In [ ]:
best = YOLO(str(LOCAL_WEIGHTS))
metrics = best.val(data=str(DATA_DIR))
print(f'Top-1 Accuracy : {metrics.top1*100:.2f}%')
print(f'Top-5 Accuracy : {metrics.top5*100:.2f}%')

## Step 9 — Training Curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

csv_path = Path(PROJECT) / RUN / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for col, color, lbl in [('train/loss','royalblue','Train'), ('val/loss','coral','Val')]:
    if col in df: ax1.plot(df['epoch'], df[col], color=color, label=lbl)
ax1.set(title='Loss', xlabel='Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
for col, color, lbl in [('metrics/accuracy_top1','seagreen','Top-1'), ('metrics/accuracy_top5','teal','Top-5')]:
    if col in df: ax2.plot(df['epoch'], df[col], color=color, label=lbl)
ax2.set(title='Accuracy', xlabel='Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Step 10 — Test Inference

In [ ]:
import random, matplotlib.pyplot as plt, matplotlib.image as mpimg

if VALID_DIR and VALID_DIR.exists():
    val_classes = [d for d in VALID_DIR.iterdir() if d.is_dir()]
    test_imgs = []
    for cls_dir in random.sample(val_classes, min(6, len(val_classes))):
        imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.JPG'))
        if imgs: test_imgs.append(random.choice(imgs))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flatten(), test_imgs):
        r    = best.predict(str(img_path), verbose=False)[0]
        pred = r.names[int(r.probs.top1)]
        conf = float(r.probs.top1conf)
        true = img_path.parent.name
        ax.imshow(mpimg.imread(str(img_path)))
        ax.set_title(
            f'True: {true[:28]}\nPred: {pred[:28]}\nConf: {conf:.3f}',
            fontsize=8, color='green' if true==pred else 'red'
        )
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('inference_samples.png', dpi=120)
    plt.show()
else:
    print('No validation directory found.')

## Step 11 — Export ONNX

In [ ]:
onnx_path = best.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True)
print(f'ONNX exported: {onnx_path}')

## Step 12 — Save Weights to Drive & Download

In [ ]:
import shutil

try: WEIGHTS_OUTPUT_DIR
except NameError:
    from pathlib import Path
    try:
        import google.colab; IN_COLAB=True
        WEIGHTS_OUTPUT_DIR = Path('/content/drive/MyDrive/niko-ai/ai-model/weights')
    except ImportError:
        IN_COLAB=False
        WEIGHTS_OUTPUT_DIR = Path('c:/Jayesh/projects/miko/ai-model/weights')
    WEIGHTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dest_pt = WEIGHTS_OUTPUT_DIR / 'best.pt'
shutil.copy(str(LOCAL_WEIGHTS), str(dest_pt))
print(f'best.pt  -> {dest_pt}')

onnx_src = Path(str(LOCAL_WEIGHTS).replace('best.pt', 'best.onnx'))
if onnx_src.exists():
    shutil.copy(str(onnx_src), str(WEIGHTS_OUTPUT_DIR / 'best.onnx'))
    print(f'best.onnx -> {WEIGHTS_OUTPUT_DIR / "best.onnx"}')

if IN_COLAB:
    from google.colab import files
    print('\nDownloading best.pt to your computer...')
    files.download(str(LOCAL_WEIGHTS))

print(f'\nNext: place best.pt at backend/ai-model/weights/best.pt')

## Done

Copy `best.pt` to `backend/ai-model/weights/best.pt` and start the backend.
